In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("messy_titanic_dataset-2.csv")
df.shape
df.dtypes
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,21,1,1,"Brown, Mrs. Ellen",FEMALE,16.1,0,0,A/5 174239,95.0755,B72,S
1,516,1,3,"Phillips, Col. Alfred",m,3.9,0,0,PC 250492,11.1378,NaN,S
2,427,1,1,"Stewart, Mrs. Rose",FEMALE,NaN,4,0,A/5 80051,191.2331,B88,Southampton
3,14,1,1,"Phillips, Mr. Herbert",Male,3.2,0,2,STON/O 337048,138.6745,C112,s
4,147,0,3,"Mitchell, Mr. Herbert",M,44.4,1,0,A/5 95589,4.3798,NaN,Queenstown


In [4]:
#Data quality report (before)
def data_quality_report(df, label):
    report = pd.DataFrame({
        "dtype": df.dtypes,
        "nulls": df.isnull().sum(),
        "null_pct": (df.isnull().mean() * 100).round(2)
    })
    print(f"--- {label} ---")
    print(report)
    print("Duplicate rows:", df.duplicated().sum())
    print("Duplicate PassengerId:", df['PassengerId'].duplicated().sum())
    return report

report_before = data_quality_report(df, "BEFORE")

--- BEFORE ---
             dtype  nulls  null_pct
PassengerId  int64      0      0.00
Survived     int64      0      0.00
Pclass       int64      0      0.00
Name           str      4      0.43
Sex            str      0      0.00
Age            str    185     19.98
SibSp        int64      0      0.00
Parch        int64      0      0.00
Ticket         str      0      0.00
Fare           str      9      0.97
Cabin          str    743     80.24
Embarked       str      4      0.43
Duplicate rows: 27
Duplicate PassengerId: 35


In [5]:
#Standardize categorical/text formatting
# Sex: collapse all variants to Male/Female
df['Sex'] = df['Sex'].str.strip().str.lower().map({
    'm': 'Male', 'male': 'Male',
    'f': 'Female', 'female': 'Female'
})

# Embarked: collapse full names + case variants to single-letter codes
embarked_map = {
    's': 'S', 'southampton': 'S',
    'c': 'C', 'cherbourg': 'C',
    'q': 'Q', 'queenstown': 'Q'
}
df['Embarked'] = df['Embarked'].str.strip().str.lower().map(embarked_map)

# Fare: strip "$" and convert to numeric
df['Fare'] = df['Fare'].astype(str).str.replace('$', '', regex=False)
df['Fare'] = pd.to_numeric(df['Fare'], errors='coerce')

# Age: strip " yrs" and convert to numeric
df['Age'] = df['Age'].astype(str).str.replace(' yrs', '', regex=False)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

In [6]:
#(outlier/data-type cleanup)
# Age can't be negative or absurdly high — treat as invalid, set to NaN
df.loc[(df['Age'] < 0) | (df['Age'] > 100), 'Age'] = np.nan

In [7]:
#Missing data handling (per column, with justification)
# Age: numeric, moderately skewed -> median imputation (robust to outliers)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Fare: numeric, right-skewed -> median imputation
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# Embarked: categorical, few missing -> mode imputation
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Cabin: >75% missing -> too sparse to impute meaningfully, keep as
# "Unknown" category rather than dropping the column/rows
df['Cabin'] = df['Cabin'].fillna('Unknown')

# Name: identifier field, can't be imputed -> drop the few rows missing it
df = df.dropna(subset=['Name'])

In [8]:
#Duplicate removal
dupes_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {dupes_before} duplicate rows")

# Also check duplicate PassengerId (same person, different row)
id_dupes = df['PassengerId'].duplicated().sum()
df = df.drop_duplicates(subset='PassengerId', keep='first')
print(f"Removed {id_dupes} duplicate PassengerId rows")

Removed 35 duplicate rows
Removed 0 duplicate PassengerId rows


In [9]:
#Outlier detection (IQR method on Fare)
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR

outliers = df[(df['Fare'] < lower) | (df['Fare'] > upper)]
print(f"Fare outliers found: {len(outliers)}")

# Decision: cap rather than remove, to preserve sample size
df['Fare'] = df['Fare'].clip(lower, upper)

Fare outliers found: 92


In [10]:
#final dtype correction
df['PassengerId'] = df['PassengerId'].astype(str)
df['Survived'] = df['Survived'].astype(int)
df['Pclass'] = df['Pclass'].astype(int)
df['Age'] = df['Age'].astype(float)
df['Fare'] = df['Fare'].astype(float)
df['SibSp'] = df['SibSp'].astype(int)
df['Parch'] = df['Parch'].astype(int)

In [11]:
#Before/after summary table
report_after = data_quality_report(df, "AFTER")

summary = pd.DataFrame({
    "rows": [926, len(df)],
    "nulls_total": [report_before['nulls'].sum(), report_after['nulls'].sum()],
    "duplicates": [dupes_before, df.duplicated().sum()]
}, index=["Before", "After"])
summary

--- AFTER ---
               dtype  nulls  null_pct
PassengerId      str      0       0.0
Survived       int64      0       0.0
Pclass         int64      0       0.0
Name             str      0       0.0
Sex              str      0       0.0
Age          float64      0       0.0
SibSp          int64      0       0.0
Parch          int64      0       0.0
Ticket           str      0       0.0
Fare         float64      0       0.0
Cabin            str      0       0.0
Embarked         str      0       0.0
Duplicate rows: 0
Duplicate PassengerId: 0


,rows,nulls_total,duplicates
Before,926,945,35
After,887,0,0


In [12]:
#Save cleaned dataset
df.to_csv("titanic_cleaned.csv", index=False)
print("Saved cleaned dataset.")

Saved cleaned dataset.
